In [0]:
%run "../_shared/02_football_data_org_client"

In [0]:
import logging
import os


logging.basicConfig(
    level=logging.INFO,
    format=(
        "%(asctime)s | "
        "%(levelname)s | "
        "%(message)s"
    ),
)

logger = logging.getLogger(__name__)

In [0]:
TOP_5_COMPETITIONS = {
    "PL": "premier_league",
    "PD": "la_liga",
    "BL1": "bundesliga",
    "SA": "serie_a",
    "FL1": "ligue_1",
}


SEASON_START_YEARS = [
    2021,
    2022,
    2023,
    2024,
    2025,
]


BASE_URL = "https://api.football-data.org/v4"

RAW_BASE_PATH = (
    "/Volumes/football_data_platform/raw/source_files/"
    "football_data_org/matches"
)


API_TOKEN = dbutils.secrets.get(
    scope="football-api",
    key="api-token",
)

In [0]:
def validate_matches_payload(
    payload: dict,
    expected_competition_code: str,
) -> None:
    """
    Validate basic business-level properties of a matches response.
    """

    competition = payload.get(
        "competition",
        {},
    )

    returned_competition_code = competition.get(
        "code"
    )

    if returned_competition_code != expected_competition_code:
        raise FootballDataOrgError(
            "Unexpected competition returned by API. "
            f"Requested '{expected_competition_code}', "
            f"received '{returned_competition_code}'."
        )

    matches = payload.get(
        "matches"
    )

    if not isinstance(matches, list):
        raise FootballDataOrgError(
            "Expected 'matches' to be a list."
        )

    if not matches:
        raise FootballDataOrgError(
            "football-data.org returned an empty matches list "
            f"for competition '{expected_competition_code}'."
        )

    result_set = payload.get(
        "resultSet",
        {},
    )

    expected_count = result_set.get(
        "count"
    )

    if (
        expected_count is not None
        and expected_count != len(matches)
    ):
        raise FootballDataOrgError(
            "Match count does not match returned matches. "
            f"resultSet.count={expected_count}, "
            f"matches={len(matches)}"
        )

In [0]:
def extract_matches(
    client: FootballDataOrgClient,
    competitions: dict[str, str],
    output_directory: str,
) -> dict[str, int]:
    """
    Extract currently available matches for configured competitions.

    Each competition response is stored as one raw JSON file.

    Existing files are skipped to keep reruns idempotent.
    """

    extracted_count = 0
    skipped_count = 0
    failed_count = 0

    for competition_code, competition_name in competitions.items():

        file_name = (
            f"{competition_code.lower()}_"
            f"{competition_name}.json"
        )

        file_path = os.path.join(
            output_directory,
            file_name,
        )



        logger.info(
            "Extracting matches: %s",
            competition_code,
        )

        try:

            payload = client.get_competition_matches(
                competition_code=competition_code,
            )

            validate_matches_payload(
                payload=payload,
                expected_competition_code=competition_code,
            )

            write_json_atomic(
                data=payload,
                file_path=file_path,
            )

            extracted_count += 1

            result_set = payload.get(
                "resultSet",
                {},
            )

            logger.info(
                "Saved matches data: "
                "%s | matches=%s | played=%s",
                competition_code,
                result_set.get("count"),
                result_set.get("played"),
            )

        except FootballDataOrgRateLimitError:

            logger.exception(
                "Rate limit reached while extracting %s. "
                "Stopping ingestion.",
                competition_code,
            )

            raise

        except FootballDataOrgError:

            failed_count += 1

            logger.exception(
                "Failed to extract matches: %s",
                competition_code,
            )

        except Exception:

            failed_count += 1

            logger.exception(
                "Unexpected error while extracting matches: %s",
                competition_code,
            )

    return {
        "extracted": extracted_count,
        "skipped": skipped_count,
        "failed": failed_count,
    }

In [0]:
with FootballDataOrgClient(
    api_token=API_TOKEN,
    base_url=BASE_URL,
    min_request_interval_seconds=6.5,
    timeout_seconds=30,
) as client:

    ingestion_summary = extract_matches(
        client=client,
        competitions=TOP_5_COMPETITIONS,
        output_directory=RAW_BASE_PATH,
    )

In [0]:
logger.info(
    "Matches ingestion completed. "
    "Extracted: %s | Skipped: %s | Failed: %s",
    ingestion_summary["extracted"],
    ingestion_summary["skipped"],
    ingestion_summary["failed"],
)

if ingestion_summary["failed"] > 0:
    raise RuntimeError(
        "Matches ingestion completed with failures. "
        f"Failed extractions: "
        f"{ingestion_summary['failed']}"
    )

ingestion_summary